In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'google-generativeai>=0.8.0',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'pyarrow>=16.0.0',
    'faker>=24.0.0',
], check=True)

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/urdu-asr-pipelines')

from shared.secrets import load_secrets
from shared.workflow_kernel import WorkflowKernel
from shared.gemini_rate_limiter import GeminiRateLimiter

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/urdu-asr-pipelines/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = 'run_20260507_001'
SESSION_ID   = 'cpu_label_01'
SESSION_TYPE = 'cpu_label'
SHARD_KEY    = 'cpu'

SECRETS = load_secrets(require_gemini=True)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = None,
    vram_limit_gb    = 0.0,
    session_max_hours = 8.5,
)
kernel.start()

GEMINI_KEY    = kernel.get_gemini_key(SECRETS)
GEMINI_LIMITER = GeminiRateLimiter(rpm_limit=14)

import google.generativeai as genai
genai.configure(api_key=GEMINI_KEY)
GEMINI_MODEL = genai.GenerativeModel('gemini-2.5-flash')

print(f'[session] {SESSION_ID} started — run={RUN_ID}')
print(f'[gemini] using key slot for session {SESSION_ID}')

In [ ]:
stages_to_run = ['p2a', 'p3a', 'p3b', 'p4a', 'p4b']

for stage in stages_to_run:
    if kernel.session_expiring:
        print(f'[session] expiring — skipping {stage}')
        break
    kernel.check_session_time()

    started_at = kernel.log_stage_start(stage)
    try:
        if stage == 'p2a':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_2_intent/p2a_label.ipynb').read())
        elif stage == 'p3a':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_3_dialogue/p3a_generate.ipynb').read())
        elif stage == 'p3b':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_3_dialogue/p3b_augment.ipynb').read())
        elif stage == 'p4a':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_4_episodes/p4a_dummy_env.ipynb').read())
        elif stage == 'p4b':
            exec(open('/kaggle/input/urdu-asr-pipelines/pipeline_4_episodes/p4b_generate.ipynb').read())
        kernel.log_stage_end(stage, started_at)
        print(f'[session] {stage} completed')
    except Exception as e:
        kernel.log_stage_end(stage, started_at, error=str(e))
        print(f'[session] {stage} failed: {e}')
        raise

kernel.stop()
print('[session] cpu_label session complete')